In [0]:
from pyspark.sql.functions import *

data = [
    (0, "Y", "N"),
    (1, "Y", "Y"),
    (2, "N", "Y"),
    (3, "Y", "Y"),
    (4, "N", "N")
]

df = spark.createDataFrame(data, schema=["product_id", "low_fats", "recyclable"])
display(df)

In [0]:
df = df.filter((col("low_fats") == "Y") & (col("recyclable") == "Y")).select("product_id")
display(df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
data = [
    (1, 100),
    (2, 200),
    (3, 300)
]

df = spark.createDataFrame(data, schema=["id", "salary"])
# df = df.withColumn("rnk",dense_rank().over(Window.orderBy(col("salary").desc()))).filter(col("rnk")==2).select("id").distinct()
# display(df)
highest_salary = df.agg(max("salary").alias("max_salary")).collect()[0]["max_salary"]

# Step 2: Filter out the highest salary and get max of remaining
second_highest_salary_df = df.filter(col("salary") != highest_salary).orderBy(col("salary").desc()).limit(1).select("id")
display(second_highest_salary_df)




In [0]:
data = [
    (1, 3.50),
    (2, 3.65),
    (3, 4.00),
    (4, 3.85),
    (5, 4.00),
    (6, 3.65)
]

df = spark.createDataFrame(data, schema=["id", "score"])
df = df.withColumn("rnk",dense_rank().over(Window.orderBy(col("score").desc()))).select("score","rnk").orderBy(col("rnk"))
display(df)

In [0]:
data = [
    (1, 1),
    (2, 1),
    (3, 1),
    (4, 2),
    (5, 1),
    (6, 2),
    (7, 2)
]

df = spark.createDataFrame(data, schema=["id", "num"])
df = df.withColumn("prev",lag("num",1,0).over(Window.orderBy(col("id")))).withColumn("prev2",lag("num",2,0).over(Window.orderBy(col("id")))).filter((col("num")==col("prev")) & (col("num")==col("prev2"))).select("num").distinct()
display(df)


In [0]:
data = [
    (1, "a@b.com"),
    (2, "c@d.com"),
    (3, "a@b.com")
]

df = spark.createDataFrame(data, ["id", "email"])
df = df.groupBy("email").agg(count("email").alias("cnt")).filter(col("cnt")>1).select("email")
display(df)

In [0]:
customers_data = [
    (1, "Joe"),
    (2, "Henry"),
    (3, "Sam"),
    (4, "Max")
]
orders_data = [
    (1, 3),
    (2, 1)
]

# Create DataFrames
c_df = spark.createDataFrame(customers_data, ["id", "name"])
o_df = spark.createDataFrame(orders_data, ["id", "customerId"])

df = c_df.join(o_df,c_df.id==o_df.customerId,"leftanti").select("name")
display(df)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window

# Initialize Spark

# Sample data
employee_data = [
    (1, "Joe", 70000, 1),
    (2, "Jim", 90000, 1),
    (3, "Henry", 80000, 2),
    (4, "Sam", 60000, 2),
    (5, "Max", 90000, 1)
]

department_data = [
    (1, "IT"),
    (2, "Sales")
]

# Create DataFrames
e = spark.createDataFrame(employee_data, ["id", "name", "salary", "departmentId"])
d = spark.createDataFrame(department_data, ["id", "name"])

# Alias DataFrames to avoid ambiguity
# e = employee_df.alias("e")
# d = department_df.alias("d")

# Join employee with department
joined_df = e.join(d, e.departmentId == d.id)

# Define window to rank employees by salary in each department
window_spec = Window.partitionBy("d.name").orderBy(col("e.salary").desc())

# Add rank column
ranked_df = joined_df.withColumn("rank", dense_rank().over(window_spec))

# Filter top-ranked (highest paid) employees per department
result_df = ranked_df.filter(col("rank") == 1).select(
    col("d.name").alias("Department"),
    col("e.name").alias("Employee"),
    col("e.salary").alias("Salary")
)

result_df.show()


In [0]:
employee_data = [
    (1, "Joe", 85000, 1),
    (2, "Henry", 80000, 2),
    (3, "Sam", 60000, 2),
    (4, "Max", 90000, 1),
    (5, "Janet", 69000, 1),
    (6, "Randy", 85000, 1),
    (7, "Will", 70000, 1)
]

# Department data
department_data = [
    (1, "IT"),
    (2, "Sales")
]

# Create DataFrames
employee_df = spark.createDataFrame(employee_data, ["id", "name", "salary", "departmentId"])
department_df = spark.createDataFrame(department_data, ["id", "name"])

e = employee_df.alias("e")
d = department_df.alias("d")

df = e.join(d,e.departmentId==d.id).withColumn("rnk",dense_rank().over(Window.partitionBy("d.name").orderBy(col("e.salary").desc()))).filter(col("rnk")<4).select(col("d.name").alias("Department"),col("e.name").alias("Employee"),col("e.salary").alias("Salary")).orderBy(col("d.name"),col("e.salary").desc())
display(df)

In [0]:
# Trip data
trips_data = [
    (1, 1, 10, 1, "completed", "2013-10-01"),
    (2, 2, 11, 1, "cancelled_by_driver", "2013-10-01"),
    (3, 3, 12, 6, "completed", "2013-10-01"),
    (4, 4, 13, 6, "cancelled_by_client", "2013-10-01"),
    (5, 1, 10, 1, "completed", "2013-10-02"),
    (6, 2, 11, 6, "completed", "2013-10-02"),
    (7, 3, 12, 6, "completed", "2013-10-02"),
    (8, 2, 12, 12, "completed", "2013-10-03"),
    (9, 3, 10, 12, "completed", "2013-10-03"),
    (10, 4, 13, 12, "cancelled_by_driver", "2013-10-03")
]

# Users data
users_data = [
    (1, "No", "client"),
    (2, "Yes", "client"),
    (3, "No", "client"),
    (4, "No", "client"),
    (10, "No", "driver"),
    (11, "No", "driver"),
    (12, "No", "driver"),
    (13, "No", "driver")
]

# Create DataFrames
trips_df = spark.createDataFrame(trips_data, ["id", "client_id", "driver_id", "city_id", "status", "request_at"])
users_df = spark.createDataFrame(users_data, ["users_id", "banned", "role"])

clients_df = users_df.filter(col("role")=="client").filter(col("banned")=="No")
drivers_df = users_df.filter(col("role")=="driver").filter(col("banned")=="No")
df = trips_df.join(clients_df,trips_df.client_id==clients_df.users_id,"inner")
df = df.join(drivers_df,trips_df.driver_id==drivers_df.users_id,"inner")
df = df.groupBy("request_at").agg(round(sum(when(col("status").like("cancelled%"),1).otherwise(0))/count(col("status")),2).alias("total"))
    
display(df)